# DreamBooth による Stable Diffusion のファインチューニング

## 学習目標
- Keras CV を介して事前学習済みの Stable Diffusion を使用する方法を学ぶ
- DreamBooth 手法を使用して Stable Diffusion をファインチューニングする方法を学ぶ

## はじめに

このラボでは、わずか 3〜5 枚の画像で新しい視覚的概念を Stable Diffusion に学習させるファインチューニング手法である DreamBooth を実装します。

DreamBooth は、Ruiz らによる論文 [DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation](https://arxiv.org/abs/2208.12242) で提案されました。<br>
このラボでは、拡散モデル（diffusion models）に関する基本的な知識があることを前提としています。まだ知識がない場合は、まず [diffusion_model.ipynb](./diffusion_model.ipynb) を参照してください。

## Keras CV での Stable Diffusion
DreamBooth 自体はテキスト条件付き画像生成モデル全般に適用できますが、このラボでは Stable Diffusion をファインチューニングします。

Stable Diffusion は、論文 ["High-Resolution Image Synthesis with Latent Diffusion Models"](https://arxiv.org/abs/2112.10752) で提案されました。<br>
この論文で提案されている潜在拡散モデル（LDM: Latent Diffusion Model）は、画像のピクセル空間ではなく、画像の潜在空間に対して拡散モデルを適用することを提案しています。潜在空間は、別途事前学習された[変分オートエンコーダ（VAE）](./variational_autoencoder.ipynb) を使用して得られます。

<img width="1084" alt="image" src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/6fced434-99b7-49aa-8828-ed760b519f63">

テキストプロンプトから簡単に画像を生成できるオープンソース実装は複数存在しますが、[KerasCV](https://keras.io/keras_cv/) の実装にはいくつかの明確な利点があります。<br>
これらには [XLA（Accelerated Linear Algebra）](https://www.tensorflow.org/xla) コンパイルと[混合精度（mixed precision）](https://www.tensorflow.org/guide/mixed_precision) のサポートが含まれ、最先端の生成速度を実現します。

Keras CV の Stable Diffusion 実装を使用して画像を生成してみましょう。

**初回実行時には数分かかります。より詳細なパフォーマンス比較については、[こちらのガイド](https://keras.io/guides/keras_cv/generate_images_with_stable_diffusion/) を参照してください。**

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")



**演習**: 
- [こちらのガイド](https://keras.io/guides/keras_cv/generate_images_with_stable_diffusion/) を参考に `StableDiffusionV2` をインスタンス化してください。
- 独自のプロンプトを記述し、Stable Diffusion の `text_to_image()` 関数を呼び出して画像を生成してください。

In [ ]:
import keras_cv
import matplotlib.pyplot as plt
from tensorflow import keras

# 混合精度を設定
keras.mixed_precision.set_global_policy("mixed_float16")

# TODO: Stable Diffusion モデルをインスタンス化
stable_diffusion_model = ...

# モデルを確実にウォームアップ
# TODO: プロンプトを記述し、インスタンス化した Stable Diffusion モデルで画像を生成
images = ...

plt.figure(figsize=(20, 20))
for i in range(len(images)):
    ax = plt.subplot(1, len(images), i + 1)
    plt.imshow(images[i])
    plt.axis("off")

良さそうですね。しかし、この画像生成モデルをもう少し「パーソナライズ」してみたいところです。

次のセクションで DreamBooth を使って試してみましょう。

## DreamBooth ファインチューニング

DreamBooth は、ある意味でテキスト条件付き拡散モデルの従来のファインチューニング手法と似ていますが、比較的大きなデータセットを使用して[事前学習済みの重みを特定のドメインに向けて完全に調整する](https://keras.io/examples/generative/finetune_stable_diffusion/) のとは異なり、モデルに新しい概念を学習させ、**3〜5 枚の画像**でモデルを「パーソナライズ」するように設計されています。

DreamBooth は、一意の識別子（このノートブックでは `hta`）をターゲット概念（例: あなたの愛犬）と紐付けることで、事前学習で学習した内容を忘れることなく新しい概念を学習します。<br>
トレーニング後、その一意の識別子を使用して、さまざまなシーンに文脈化された被写体の新しい写実的な画像を合成できます。

DreamBooth の用途は非常に多彩です。お気に入りの視覚的概念を Stable Diffusion に教えることで、次のようなことが可能になります:

**オブジェクトを興味深い方法で再文脈化する:**

  ![](https://i.imgur.com/4Da9ozw.png)

**根本的な視覚的概念のアート風レンダリングを生成する:**

  ![](https://i.imgur.com/nI2N8bI.png)


その他にも多くの応用があります。詳細については、ぜひオリジナルの [DreamBooth 論文](https://arxiv.org/abs/2208.12242) をご覧ください。

### セットアップ
Stable Diffusion のファインチューニングには、少なくとも 24 GB の VRAM を備えた GPU が必要です。<br>
ただし、このノートブックはファインチューニングに Agent Platform Training を使用するように設計されているため、手元のマシンにそのような大容量 GPU は不要です。

それでは、まず Google Cloud の `PROJECT` と GCS の `BUCKET` を設定しましょう。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
%env PROJECT=$PROJECT
%env BUCKET=$BUCKET

また、クラウドトレーニング用の Python ファイルを構築したいと同時に、このノートブックのカーネル上でコードを実行して各ステップをインタラクティブに確認したいと考えています。

ここでは、その両方を行うためのヘルパーマジック関数を定義します。各セルの先頭に `%%write_and_run` と記述することで、ファイルの書き込みとコードの実行を同時に行うことができます。<br>
また、`-a` オプションを使用すると、既存のファイルにコードを追記できます。

In [ ]:
from IPython.core.magic import register_cell_magic


@register_cell_magic
def write_and_run(line, cell):
    argz = line.split()
    file = argz[-1]
    mode = "w"
    if len(argz) == 2 and argz[0] == "-a":
        mode = "a"
    with open(file, mode) as f:
        f.write(cell)
    get_ipython().run_cell(cell)


TRAINING_APP_FOLDER = "dreambooth_model"

### ライブラリのインポート

In [ ]:
%%write_and_run {TRAINING_APP_FOLDER}/dreambooth_trainer.py

import os
import math
import time


import keras_cv
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow.experimental.numpy as tnp

from tensorflow import keras

from google.cloud import aiplatform

### インスタンス画像とクラス画像

DreamBooth では、「事前分布の保持（Prior Preservation）」と呼ばれる手法を使用してトレーニング手順を適切に誘導し、導入する視覚的概念の事前セマンティクス（例: 'dog' クラスの概念）をファインチューニングされたモデルが保持できるようにします。「事前分布の保持」の考え方の詳細については、[こちらのドキュメント](https://dreambooth.github.io/) を参照してください。

ここで、DreamBooth に特有のいくつかの重要な用語を紹介する必要があります:

* **インスタンス（Instance）**: 学習させたい視覚的概念を表す画像です。画像の枚数は通常わずか 3〜5 枚です。通常、これらの画像は自分で収集します。プロンプトについては、「インスタンスプロンプト」を作成する際に、クラス名の前に一意の識別子を付加します。この例では、この一意の識別子として `"hta"` を使用します。「インスタンス画像」を最もよく説明するプロンプトを構築する必要があります。プロンプトの例としては、`f"a photo of a {unique_id} {unique_class}"` のようになります。したがって、今回の例では `"a photo of a hta dog"`（hta 犬の写真）となります。
* **クラス（事前分布 / Prior）**: DreamBooth トレーニングで「事前分布の保持」を使用するために、「クラスプロンプト」を使用して生成された画像です。「事前分布の保持」は、事前学習で獲得した一般的なクラス表現（「犬」クラスや「人間」クラスなど）をモデルが忘れてしまうのを防ぐために使用されます。この例では「dog」を使用します。ファインチューニング前の事前学習済みモデルを活用して、これらのクラス画像を生成します。通常、200〜300 枚のクラス画像があれば十分です。プロンプトは `"a photo of a dog"`（犬の写真）のようになります。

**注**: 事前分布の保持は DreamBooth で使用されるオプションの手法ですが、生成画像の品質向上にほぼ常に役立ちます。

クラスの事前分布画像を生成するには、クラスプロンプトを指定して事前学習済み Stable Diffusion モデルを呼び出すだけです。

```python
class_prompt = "a photo of a dog"
images = model.text_to_image(class_prompt)
```

このサンプルの実行時間を短縮するため、クラス事前分布画像はすでに生成されています。完全なコードについては、[こちらのコード](../dreambooth_utils/prior_generator.ipynb) を参照してください。

これらのデータは一般公開されている GCS バケットでホストされています。

In [ ]:
!gcloud storage ls gs://asl-public-data/data/dreambooth_dog/instance/* | head -n 3

In [ ]:
!gcloud storage ls gs://asl-public-data/data/dreambooth_dog/class/* | head -n 3

### 画像の読み込みと可視化

`tf.data` API を使用してデータローダーを設定し、インスタンス画像とクラス画像の両方からいくつかのサンプルを確認してみましょう。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

instance_images = tf.data.Dataset.list_files("gs://asl-public-data/data/dreambooth_dog/instance/*")
class_images = tf.data.Dataset.list_files("gs://asl-public-data/data/dreambooth_dog/class/*")

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

def decode_jpg(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    return img

そして、このユーティリティ関数を使って画像をプロットします。

In [ ]:
def plot_images(path_ds, num=5):
    f, ax = plt.subplots(1, num, figsize=(16, num))
    for idx, img in enumerate(path_ds.map(decode_jpg).take(num)):
        ax[idx].imshow(tf.keras.preprocessing.image.array_to_img(img))
        ax[idx].axis("off")

**インスタンス画像**: これらの美しいインスタンス写真は [Alvan Nee](https://unsplash.com/@alvannee) 氏によって撮影されたものです。

In [ ]:
plot_images(instance_images)

**クラス画像（事前分布 / Prior）**: これらの画像は事前学習済みモデル自体によって生成されたものです。

In [ ]:
plot_images(class_images)

### データセットの準備
トレーニングでは、各ステップでインスタンスデータとクラスデータのペアを渡します。<br>
ただし、これらのデータセットの長さが異なるため（インスタンス: 5、クラス: 200）、インスタンスデータセットを繰り返して長さを揃えましょう。

これを行うには、`tf.data.Dataset` の `.repeat()` メソッドを呼び出すだけです。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

instance_len = instance_images.cardinality() # 5
class_len = class_images.cardinality() # 200
ds_len = instance_len * (class_len // instance_len)

instance_images = instance_images.repeat(class_len // instance_len)
class_images = class_images.take(ds_len)

#### プロンプトの準備
各データセットには、プロンプトと画像という 2 つのコンポーネントが含まれています。両方を前処理しましょう。

プロンプトデータセットはシンプルです。
各ステップでインスタンスプロンプト（`"a photo of a hta dog"`）とクラスプロンプト（`"a photo of a dog"`）を繰り返すだけです。

Keras CV には事前学習済みの `SimpleTokenizer` が含まれているため、これを使用してプロンプトをトークン化できます。

プロンプトトークンの長さ（`MAX_PROMPT_LENGTH`）とパディングトークンのインデックス（`PADDING_TOKEN`）は、このトークナイザとテキストエンコーダに由来することに注意してください。<br>
Keras CV の実装では、CLIP テキストエンコーダが使用されています。

テキストエンコーダとこれらのハイパーパラメータの詳細については、[ソースコード](https://github.com/keras-team/keras-cv/blob/master/keras_cv/models/stable_diffusion/text_encoder.py) を参照してください。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

# 参考: https://github.com/keras-team/keras-cv/blob/master/keras_cv/models/stable_diffusion/text_encoder.py
PADDING_TOKEN = 49407
MAX_PROMPT_LENGTH = 77

# 画像ごとにプロンプト／キャプションを繰り返します。
unique_id = "hta"
class_label = "dog"

instance_prompt = f"a photo of a {unique_id} {class_label}"
class_prompt = f"a photo of a {class_label}"

# トークナイザを読み込む
tokenizer = keras_cv.models.stable_diffusion.SimpleTokenizer()

# トークン化してパディングするメソッド
def process_text(caption):
    tokens = tokenizer.encode(caption)
    tokens = tokens + [PADDING_TOKEN] * (MAX_PROMPT_LENGTH - len(tokens))
    return tokens

tokenized_instance_prompt = process_text(instance_prompt)
tokenized_class_prompt = process_text(class_prompt)

instance_prompt_ds = \
    tf.data.Dataset.from_tensor_slices([tokenized_instance_prompt]).repeat(ds_len)
class_prompt_ds = \
    tf.data.Dataset.from_tensor_slices([tokenized_class_prompt]).repeat(ds_len)

In [ ]:
print(f"{'='*10} Instance Prompt Token {'='*10}")
for d in instance_prompt_ds.take(1):
    print(d)

print(f"{'='*10} Class Prompt Token {'='*10}")
for d in class_prompt_ds.take(1):
    print(d)

#### 画像の準備

画像については、標準的な前処理といくつかのデータ拡張（Augmentation）を設定します。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

resolution = 512
auto = tf.data.AUTOTUNE

augmenter = keras.Sequential(
    layers=[
        keras_cv.layers.CenterCrop(resolution, resolution),
        keras_cv.layers.RandomFlip(),
        keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1),
    ]
)

def process_image(image_path, tokenized_text):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_png(image, 3)
    image = tf.image.resize(image, (resolution, resolution))
    return image, tokenized_text


def apply_augmentation(image_batch, tokenized_texts):
    return augmenter(image_batch), tokenized_texts

#### データセットの組み立て

`tf.data.Dataset.zip()` メソッドを使用して、プロンプトと画像のデータセットを組み立てましょう。

最終的なデータセットは、2 つのキー（`"***_images"` と `"***_texts"`）を持つ辞書で構成されていることに注意してください。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

def prepare_dict(instance_only=True):
    def fn(image_batch, tokenized_texts):
        if instance_only:
            batch_dict = {
                "instance_images": image_batch,
                "instance_texts": tokenized_texts,
            }
            return batch_dict
        else:
            batch_dict = {
                "class_images": image_batch,
                "class_texts": tokenized_texts,
            }
            return batch_dict

    return fn


def assemble_dataset(
    image_ds, tokenized_ds, instance_only=True, batch_size=1
):
    dataset = tf.data.Dataset.zip((image_ds, tokenized_ds))
    dataset = dataset.shuffle(5, reshuffle_each_iteration=True)
    dataset = dataset.map(process_image, num_parallel_calls=auto)
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(apply_augmentation, num_parallel_calls=auto)

    prepare_dict_fn = prepare_dict(instance_only=instance_only)
    dataset = dataset.map(prepare_dict_fn, num_parallel_calls=auto)
    return dataset


BATCH_SIZE = 1

instance_dataset = assemble_dataset(
    instance_images,
    instance_prompt_ds,
    batch_size=BATCH_SIZE,
)
class_dataset = assemble_dataset(
    class_images,
    class_prompt_ds,
    instance_only=False,
    batch_size=BATCH_SIZE,
)
train_dataset = tf.data.Dataset.zip((instance_dataset, class_dataset))

データセットの準備ができたので、各特徴量の形状（shape）を簡単に確認してみましょう。

In [ ]:
for d in train_dataset.take(1):
    for idx in range(len(d)):
        print(d[idx].keys())
        for key in d[idx].keys():
            print(f"shape of {key:15s}: {d[idx][key].shape}")

### DreamBooth トレーニングループ
トレーニング用のメインモデルクラスを `DreamBoothTrainer` クラスとして定義しましょう。

まず、Stable Diffusion は以下の複数のモデルで構成されていることを思い出してください:
- テキストエンコーダ（`StableDiffusionV2.text_encoder`）
- 拡散モデル（`StableDiffusionV2.diffusion_model`）
- 変分オートエンコーダ（`StableDiffusionV2.image_encoder` および `StableDiffusionV2.decoder`）

事前学習済みの重みからトレーニングを開始します。以下の `__init__` でそれらを構築していることに注目してください。<br>
テキストエンコーダと拡散モデルをファインチューニングしますが、VAE エンコーダは凍結（freeze）します。潜在空間で損失を計算できるため、トレーニングに VAE デコーダは不要です。

`train_step()` は、クラスプロンプトとインスタンスプロンプトを使用してファインチューニングプロセスが定義されるメイン関数です。<br>
クラスデータとインスタンスデータは最初に連結され、`with tf.GradientTape` 内で定義されたトレーニングプロセスに渡されます。これには大まかに以下が含まれます:
1. **プロンプトのエンコード**: 事前学習済みのテキストエンコーダが使用されます。
2. **画像のエンコード**: VAE エンコーダが使用されます。`sample_from_encoder_outputs()` 関数が VAE の潜在空間からのサンプリングを処理することに注意してください。（この考え方に馴染みがない場合は、VAE ノートブックを参照してください。）
3. **順方向拡散プロセス（Forward Diffusion Process）**: ノイズをサンプリングし、画像に追加します。
4. **逆方向拡散プロセス / 拡散モデルの予測（Reversed Diffusion Process）**: U-Net ベースの拡散モデルがノイズの混ざった画像からノイズを予測します。
5. **損失の計算**: 損失は標準的な拡散モデルと同様に単純な MSE です。`compute_loss()` 関数で定義されているように、必要に応じてクラス損失とインスタンス損失のバランスを取る重みを追加できます。

損失を計算した後、拡散モデルとテキストエンコーダに関する損失を計算し、それぞれを更新します。（VAE は凍結されたままです。）

**演習**: 上記の説明を参照して、`train_step()` と `compute_loss()` 関数を完成させてください。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

class DreamBoothTrainer(tf.keras.Model):
    def __init__(
        self,
        sd_model,
        noise_scheduler,
        max_prompt_length=77,  # デフォルトは 77
        use_mixed_precision=False,
        prior_loss_weight=1.0,
        max_grad_norm=1.0,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.sd_model = sd_model
        self.noise_scheduler = noise_scheduler
        self.max_prompt_length = max_prompt_length
        self.use_mixed_precision = use_mixed_precision
        self.prior_loss_weight = prior_loss_weight
        self.max_grad_norm = max_grad_norm

        self.diffusion_model = sd_model.diffusion_model
        self.text_encoder = sd_model.text_encoder

        self.image_encoder = sd_model.image_encoder
        self.image_encoder.trainable = False

    def compile(self, diffusion_optimizer, text_encoder_optimizer, loss):
        super().compile(loss=loss)
        self.diffusion_optimizer = diffusion_optimizer
        self.text_encoder_optimizer = text_encoder_optimizer
        
    def train_step(self, inputs):
        instance_batch = inputs[0]
        class_batch = inputs[1]

        instance_images = instance_batch["instance_images"]
        instance_text = instance_batch["instance_texts"]
        class_images = class_batch["class_images"]
        class_text = class_batch["class_texts"]

        images = tf.concat([instance_images, class_images], 0)
        tokenized_texts = tf.concat([instance_text, class_text], 0)
        batch_size = tf.shape(images)[0]
        POS_IDS = tf.convert_to_tensor(
            [list(range(self.max_prompt_length))], dtype=tf.int32
        )

        with tf.GradientTape(persistent=True) as tape:
            embedded_texts = self.text_encoder(
                [tokenized_texts, POS_IDS], training=True
            )

            # 画像を潜在空間に射影
            # TODO: 画像エンコードを定義
            latents = ...

            # 潜在変数に追加するノイズをサンプリング
            noise = tf.random.normal(tf.shape(latents))

            # 各画像のランダムなタイムステップをサンプリング
            timesteps = tnp.random.randint(
                0, self.noise_scheduler.train_timesteps, (batch_size,)
            )

            # 各タイムステップのノイズ強度に応じて潜在変数にノイズを追加
            # （これが順方向拡散プロセスです）
            noisy_latents = self.noise_scheduler.add_noise(
                tf.cast(latents, noise.dtype), noise, timesteps
            )

            # サンプリングされたノイズがターゲット
            target = noise
            timestep_embedding = tf.map_fn(
                lambda t: self.sd_model._get_timestep_embedding(t, batch_size),
                timesteps,
                fn_output_signature=tf.float32,
            )[:, 0]

            # ノイズ残差を予測して損失を計算
            # TODO: 逆方向拡散を定義
            model_pred = self.diffusion_model(...)
            
            loss = self.compute_loss(target, model_pred)
            if self.use_mixed_precision:
                loss = self.optimizer.get_scaled_loss(loss)

        # テキストエンコーダと拡散モデルのパラメータを更新
        text_trainable_vars = self.text_encoder.trainable_variables
        diffusion_trainable_vars = self.diffusion_model.trainable_variables
        
        text_gradients = tape.gradient(loss, text_trainable_vars)
        diffusion_gradients = tape.gradient(loss, diffusion_trainable_vars)

        if self.use_mixed_precision:
            text_gradients = self.optimizer.get_unscaled_gradients(text_gradients)
            diffusion_gradients = self.optimizer.get_unscaled_gradients(diffusion_gradients)
        text_gradients = [tf.clip_by_norm(g, self.max_grad_norm) for g in text_gradients]
        diffusion_gradients = [tf.clip_by_norm(g, self.max_grad_norm) for g in diffusion_gradients]

        self.text_encoder_optimizer.apply_gradients(zip(text_gradients, text_trainable_vars))
        self.diffusion_optimizer.apply_gradients(zip(diffusion_gradients, diffusion_trainable_vars))

        return {m.name: m.result() for m in self.metrics}
    
    def compute_loss(self, target, model_pred):
        # 入力の前半にインスタンスサンプル、後半にクラスサンプルがあるため、
        # それに応じて分割を行います。
        model_pred, model_pred_prior = tf.split(
            model_pred, num_or_size_splits=2, axis=0
        )
        target, target_prior = tf.split(target, num_or_size_splits=2, axis=0)

        # TODO: インスタンス損失を定義
        loss = ...

        # TODO: 事前分布損失を定義
        prior_loss = ...

        # 事前分布損失をインスタンス損失に加算
        # TODO: `self.prior_loss_weight` を使用してインスタンス損失と事前分布損失を加算
        loss = ...
        return loss

    def save_weights(
        self, filepath, overwrite=True, save_format=None, options=None
    ):
        # このメソッドをオーバーライドすることで、保存する重みをカスタマイズできます。
        # この場合、`diffusion_model` と `text_encoder` のチェックポイントを保存します。
        self.diffusion_model.save_weights(
            filepath=f"{filepath}diffusion",
            overwrite=overwrite,
            save_format=save_format,
            options=options,
        )

        self.text_encoder.save_weights(
            filepath=f"{filepath}text_encoder",
            overwrite=overwrite,
            save_format=save_format,
            options=options,
        )

### トレーナーの初期化

それでは、トレーナーを構築し、通常通りコンパイルしましょう。

単一のオプティマイザで両方をトレーニングすることも可能ですが、ここでは拡散モデル用とテキストエンコーダ用にそれぞれ 2 つのオプティマイザを定義していることに注意してください。<br>
こうすることで、学習率とスケジューリングを個別に制御できます。<br>
Stable Diffusion はハイパーパラメータに非常に敏感で、容易に過学習／過小学習を起こすため、きめ細かい制御を行うことが有効です。

**演習**: 2 つのオプティマイザを使用して `compile()` を完成させてください。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

def build_trainer(
    diffusion_learning_rate,
    text_learning_rate,
    diffusion_beta_1=0.9,
    diffusion_beta_2=0.999,
    text_beta_1=0.9,
    text_beta_2=0.999,
    diffusion_weight_decay=(1e-2,),
    text_weight_decay=(1e-2,),
    epsilon = 1e-08,
    use_mp=True # Tensor Core を搭載していない GPU を使用している場合は False に設定してください。
):
    if use_mp:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")

    dreambooth_model = keras_cv.models.StableDiffusionV2(
        img_width=resolution, img_height=resolution, jit_compile=True
    )

    dreambooth_trainer = DreamBoothTrainer(
        sd_model=dreambooth_model,
        noise_scheduler=keras_cv.models.stable_diffusion.NoiseScheduler(),
        use_mixed_precision=use_mp,
    )

    diffusion_optimizer = tf.keras.optimizers.AdamW(
        learning_rate=diffusion_learning_rate,
        weight_decay=diffusion_weight_decay,
        beta_1=diffusion_beta_1,
        beta_2=diffusion_beta_2,
        epsilon=epsilon,
    )

    text_encoder_optimizer = tf.keras.optimizers.AdamW(
        learning_rate=text_learning_rate,
        weight_decay=text_weight_decay,
        beta_1=text_beta_1,
        beta_2=text_beta_2,
        epsilon=epsilon,
    )

    dreambooth_trainer.compile(...)
    
    return dreambooth_trainer

#### Tensorboard コールバックの構築（オプション）

生成モデルのトレーニングでは、定性的評価（定量的指標ではなく生成された出力を確認すること）が非常に重要です。
この目的のために、追加の Tensorboard コールバック関数を定義して、生成出力によってトレーニングプロセスを確認すると便利です。

カスタムコールバック関数 `"ImageGenCallback"` は、各エポックの終わりに画像を生成するように定義されています。<br>
カスタムコールバック関数の詳細については、[こちらのドキュメント](https://www.tensorflow.org/guide/keras/writing_your_own_callbacks) を参照してください。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

class ImageGenCallback(tf.keras.callbacks.Callback):
    def __init__(self, tensorboard_path, prompts, num_imgs_to_gen=3):
        self.tensorboard_path = tensorboard_path
        self.prompts = prompts
        self.num_imgs_to_gen = num_imgs_to_gen

    def log_image(self, prefix, prompt, step):
        images_dreamboothed = self.model.sd_model.text_to_image(
            prompt, batch_size=self.num_imgs_to_gen
        )
        logdir = f"{self.tensorboard_path}images"
        file_writer = tf.summary.create_file_writer(logdir)

        with file_writer.as_default():
            tf.summary.image(
                f"{prefix}: '{prompt}'",
                images_dreamboothed,
                step=step,
            )

    def on_train_begin(self, logs=None):
        prefix = "0_Initial Model"
        for p in self.prompts:
            self.log_image(prefix, p, step=0)

    def on_epoch_end(self, epoch, logs=None):
        prefix = f"{epoch+1}_Epoch {epoch+1}"
        for p in self.prompts:
            self.log_image(prefix, p, step=epoch+1)

def construct_tb_callbacks(tensorboard_path, prompts, num_imgs_to_gen=3):

    tf_callback = tf.keras.callbacks.TensorBoard(
        log_dir=tensorboard_path,
        write_graph=False,
    )

    return [tf_callback, ImageGenCallback(tensorboard_path, prompts, num_imgs_to_gen)]

### トレーニング
まず、トレーニングに必要なエポック数を計算します。<br>
論文では、学習率 6e-6（Stable Diffusion の場合）で 1,000 ステップ実行すれば、ほとんどのケースで機能することが示唆されています。ただし、人間の顔のような、より複雑で繊細な被写体の場合は、より多くのステップや異なるハイパーパラメータのセットが必要になる場合があります。

ここでは、インスタンスである犬のパターンを学習できるように、少し大きめの学習率で 1,000 ステップを使用します。

In [ ]:
%%write_and_run -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

MAX_TRAIN_STEP = 1000

num_update_steps_per_epoch = ds_len // BATCH_SIZE
epochs = math.ceil(MAX_TRAIN_STEP / num_update_steps_per_epoch)

print(f"Training for {epochs} epochs.")

このセルでトレーニングを実行します。<br>
ただし、ローカルで実行したくないため、`%%write_and_run` の代わりに `%%writefile` マジックを使用して、実行せずにファイルへの書き込みのみを行います。<br>
クラウドトレーニングジョブは後で送信します。

ローカルマシンに A100 GPU が搭載されている場合は、以下のコマンドでトレーニングを実行することも可能です。（A100 ノートブックインスタンスを作成したい場合は、この [bash ファイル](../dreambooth_utils/create_a100_notebook.sh) を実行してください。）

```python
output_base_dir = "dreambooth_outputs"
tensorboard_path = f"{output_base_dir}/logs/"

!rm -rf {output_base_dir}
os.makedirs(output_base_dir)

dreambooth_trainer= build_trainer(
    diffusion_learning_rate = 8e-6,
    text_learning_rate = 6e-7,
)

%load_ext tensorboard
%tensorboard --logdir {tensorboard_path}

callback_prompts = [f"A photo of a {unique_id} {class_label}"]

dreambooth_trainer.fit(
    train_dataset,
    epochs=epochs,
    callbacks=construct_tb_callbacks(tensorboard_path, callback_prompts),
)

os.makedirs(f"{output_base_dir}/checkpoints")
dreambooth_trainer.save_weights(f"{output_base_dir}/checkpoints/")
```

In [ ]:
%%writefile -a {TRAINING_APP_FOLDER}/dreambooth_trainer.py

dreambooth_trainer= build_trainer(
    diffusion_learning_rate = 8e-6,
    text_learning_rate = 6e-7,
)

tensorboard_path = os.getenv("AIP_TENSORBOARD_LOG_DIR")

callback_prompts = [f"A photo of a {unique_id} {class_label}"]

dreambooth_trainer.fit(
    train_dataset,
    epochs=epochs,
    callbacks=construct_tb_callbacks(tensorboard_path, callback_prompts),
)

dreambooth_trainer.save_weights(os.getenv("AIP_CHECKPOINT_DIR"))

#### クラウドトレーニング
これで、トレーニングコード全体が完成しました。<br>
環境をシンプルな Docker コンテナにコンテナ化し、Artifact Registry に push します。

In [ ]:
%%writefile {TRAINING_APP_FOLDER}/Dockerfile
FROM us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-12.py310:latest
RUN pip install keras_cv==0.5.1 --upgrade --quiet

COPY . /code

WORKDIR /code

ENTRYPOINT ["python3", "dreambooth_trainer.py"]

In [ ]:
IMAGE_NAME = "dreambooth_trainer"

!gcloud builds submit --tag us-docker.pkg.dev/{PROJECT}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME} {TRAINING_APP_FOLDER}

ワーカープール仕様で `NVIDIA_TESLA_A100` を指定します。A100 GPU でのトレーニングには約 30 分かかります。

**ただし、A100 GPU は需要が高いため、リソースエラーによりトレーニング実行コマンドが失敗する場合があります。**<br>
 <span style="color:red">**リソースエラーが表示された場合でも、このセルはスキップして構いません。**</span><br>
 <span style="color:red">**同じセットアップでトレーニングを事前に実行し、結果を保存してあります。次のセクションで確認できます。**</span><br>

In [ ]:
TRAIN_CONTAINER_URI = (
    f"us-docker.pkg.dev/{PROJECT}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}"
)

TRAIN_MACHINE_TYPE = "a2-highgpu-1g"
TRAIN_ACCELERATOR_TYPE = "NVIDIA_TESLA_A100"
TRAIN_NUM_GPU = 1

RESOLUTION = 512
REGION = "us-central1"

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": TRAIN_MACHINE_TYPE,
            "accelerator_type": TRAIN_ACCELERATOR_TYPE,
            "accelerator_count": TRAIN_NUM_GPU,
        },
        "replica_count": 1,
        "disk_spec": {
            "boot_disk_type": "pd-ssd",
            "boot_disk_size_gb": 500,
        },
        "container_spec": {
            "image_uri": TRAIN_CONTAINER_URI,
            "command": [],
        },
    }
]

TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"dreambooth_training_{TIMESTAMP}"

train_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    project=PROJECT,
    location=REGION,
    worker_pool_specs=worker_pool_specs,
    base_output_dir=f"gs://{BUCKET}/dreambooth_outputs/{JOB_NAME}",
    staging_bucket=f"gs://{BUCKET}",
)

train_job.run()

### 推論

それでは、結果を見てみましょう。

A100 でトレーニングを実行できた場合は、`output_base_dir` を自身のバケットに置き換えてください。<br>
（ローカルで実行した場合はローカルパスに置き換えてください。）

In [ ]:
# 事前トレーニング済みの重み
output_base_dir = "gs://asl-public-data/model_checkpoints/dreambooth_trained"

# クラウドトレーニングの出力先パス
# output_base_dir = f"gs://{BUCKET}/dreambooth_outputs/{JOB_NAME}"

# またはローカルでのトレーニング出力パス
# output_base_dir = "dreambooth_outputs"

追加した Tensorboard コールバックのおかげで、Tensorboard を使用してファインチューニングプロセスの進行状況を確認できます。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {output_base_dir}/logs

いかがでしょうか？トレーニングプロセスにおいて、モデルがインスタンスの概念を徐々に獲得していることが確認できます。

次に、ファインチューニングされたモデルのチェックポイントを読み込みましょう。

In [ ]:
# Stable Diffusion の重みを読み込む
stable_diffusion_model.diffusion_model.load_weights(
    f"{output_base_dir}/checkpoints/diffusion"
)
stable_diffusion_model.text_encoder.load_weights(
    f"{output_base_dir}/checkpoints/text_encoder"
)

プロンプトを使用して画像を生成できます。<br>
プロンプト内で一意の識別子とクラスがどのように使用されているかに注目してください。

さまざまなプロンプト（一意の識別子とクラスラベルを追加することを忘れないでください）を自由に試して、結果がどのように変化するかを確認してみてください。

In [ ]:
def plot_inference(prompts, num_imgs_to_gen, num_steps=50):
    images_dreamboothed = stable_diffusion_model.text_to_image(
        prompt, batch_size=num_imgs_to_gen, num_steps=num_steps
    )

    plt.figure(figsize=(20, 20))
    for i in range(len(images_dreamboothed)):
        ax = plt.subplot(1, num_imgs_to_gen, i + 1)
        plt.title(prompts)
        plt.imshow(images_dreamboothed[i])
        plt.axis("off")

In [ ]:
prompt = f"A photo of a {unique_id} {class_label} in a car"

plot_inference(prompt, 3)

In [ ]:
prompt = f"A photo of a {unique_id} {class_label} in front of Eiffel Tower"
plot_inference(prompt, 3)

In [ ]:
prompt = f"A photo of a black {unique_id} {class_label}"

plot_inference(prompt, 3)

また、識別子を削除することで、モデルが犬の一般的な概念を忘れていないかどうかも確認してみましょう。

In [ ]:
prompt = f"A photo of a {class_label} in front of Eiffel Tower"
plot_inference(prompt, 3)

## 謝辞
このノートブックのコードは、[Sayak Paul 氏と Chansung Park 氏による Keras 公式チュートリアル](https://keras.io/examples/generative/dreambooth/) に基づいています。